In [ ]:
!git -C /content/ANN pull origin main

In [ ]:
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd 
import torch
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

repo_path = Path("/content/ANN")
sys.path.insert(0, str(repo_path))

from src.data_loader import load_batches
from src.model import Model
from IPython.display import display as show_table



In [ ]:
from google.colab import drive

drive_root = Path("/content/gdrive")

if not (drive_root / "MyDrive").exists():
    drive.mount("/content/gdrive")

data_path = drive_root / "MyDrive" / "ANN-data"
model_path = repo_path / "models" / "baseline_best.pth"
scaler_path = repo_path / "models" / "scaler.pkl"

for path in [data_path, model_path, scaler_path]:
    print(path, "Exists:", path.exists())

In [ ]:
batches = load_batches(data_path)

scaler = joblib.load(scaler_path)

model = Model()
model.load_state_dict(
    torch.load(
        model_path,
        map_location="cpu",
        weights_only=True
    )
)
model.eval()


In [ ]:
class_names = [
    "Ethanol",
    "Ethylene",
    "Ammonia",
    "Acetaldehyde",
    "Acetone",
    "Toluene"
]

loss_func = torch.nn.CrossEntropyLoss()
results = []

for batch_num in range(8, 11):
    batch = batches[f"batch{batch_num}"]

    X_test = scaler.transform(batch["features"])
    y_test = np.asarray(batch["labels"], dtype=np.int64) - 1

    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.int64)

    with torch.no_grad():
        logits = model(X_test)
        loss = loss_func(logits, y_test_tensor)
        predictions = logits.argmax(dim=1).numpy()

    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(
        y_test,
        predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "Batch": batch_num,
        "Samples": len(y_test),
        "Loss": loss.item(),
        "Accuracy": accuracy,
        "Macro-F1": macro_f1
    })

    print(f"\n{'=' * 12} BATCH {batch_num} {'=' * 12}")
    print(f"Samples: {len(y_test)}")
    print(f"Test loss: {loss.item():.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro-F1: {macro_f1:.4f}")

    print(classification_report(
        y_test,
        predictions,
        labels=list(range(6)),
        target_names=class_names,
        digits=3,
        zero_division=0
    ))

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=list(range(6))
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    display.plot(ax=ax, xticks_rotation=45)
    ax.set_title(f"Test Confusion Matrix — B{batch_num}")
    plt.tight_layout()
    plt.show()

In [ ]:
results_df = pd.DataFrame(results)

results_df

In [17]:
per_class_results = []

model.eval()

for batch_num in range(8, 11):
    batch = batches[f"batch{batch_num}"]

    X_test = scaler.transform(batch["features"])
    y_test = np.asarray(batch["labels"], dtype=np.int64) - 1

    X_test = torch.tensor(X_test, dtype=torch.float32)

    with torch.no_grad():
        predictions = model(X_test).argmax(dim=1).numpy()

    report = classification_report(
        y_test,
        predictions,
        labels=list(range(6)),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    for gas in class_names:
        per_class_results.append({
            "Batch": f"B{batch_num}",
            "Gas": gas,
            "Precision": report[gas]["precision"],
            "Recall": report[gas]["recall"],
            "F1-score": report[gas]["f1-score"],
            "Support": report[gas]["support"]
        })

results_df = pd.DataFrame(per_class_results)

show_table(results_df.round(3))

,Batch,Gas,Precision,Recall,F1-score,Support
0,B8,Ethanol,0.652,1.000,0.789,30.0
1,B8,Ethylene,0.566,1.000,0.723,30.0
2,B8,Ammonia,1.000,0.675,0.806,40.0
3,B8,Acetaldehyde,1.000,0.152,0.263,33.0
4,B8,Acetone,0.932,0.965,0.948,143.0
5,B8,Toluene,0.667,0.556,0.606,18.0
6,B9,Ethanol,0.372,1.000,0.542,61.0
7,B9,Ethylene,0.833,1.000,0.909,55.0
8,B9,Ammonia,1.000,1.000,1.000,100.0
9,B9,Acetaldehyde,1.000,0.200,0.333,75.0
